# 호출 그래프 분석 및 강결합 컴포넌트(SCC) 감지

이 노트북은 UnifyWeaver의 고급 코드 분석 기능을 탐구합니다:

- **호출 그래프 구축 (Call Graph Construction)** - Prolog 코드로부터 의존성 그래프 구축
- **SCC 감지** - 강결합 컴포넌트(상호 재귀) 찾기
- **패턴 분석** - 재귀 패턴 이해
- **의존성 시각화** - 서술어 간의 관계 시각화

## 학습 목표

- UnifyWeaver가 코드 구조를 분석하는 방식 이해
- 호출 그래프 구축 및 검사
- 타잔(Tarjan) 알고리즘을 사용한 상호 재귀 감지
- 코드 의존성 시각화

## 설정

UnifyWeaver 및 분석 모듈을 로드합니다.

In [ ]:
% Load initialization
['../init'].

% Load analysis modules
use_module(unifyweaver(core/advanced/call_graph)).
use_module(unifyweaver(core/advanced/scc_detection)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## 예제 1: 간단한 호출 그래프

간단한 서술어로 시작하여 호출 그래프를 구축해 보겠습니다.

In [ ]:
% Define ancestor predicate
:- dynamic ancestor/2.
:- dynamic parent/2.

% Parent facts
parent(abraham, isaac).
parent(isaac, jacob).

% Ancestor rules
ancestor(X, Y) :- parent(X, Y).
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

### 호출 그래프 구축

In [ ]:
% Build call graph for ancestor
build_call_graph([ancestor/2], _Graph),
format('Call Graph for ancestor/2:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### 의존성 분석

In [ ]:
% Get all dependencies of ancestor/2
get_dependencies(ancestor/2, _Deps),
format('Dependencies of ancestor/2: ~w~n', [_Deps]).

% Check if self-recursive
(is_self_recursive(ancestor/2) ->
    writeln('✓ ancestor/2 is self-recursive')
;
    writeln('✗ ancestor/2 is not self-recursive')
).

## 예제 2: 상호 재귀 감지

이제 짝수/홀수 예제로 상호 재귀를 감지해 보겠습니다.

In [ ]:
% Define mutually recursive predicates
:- dynamic is_even/1.
:- dynamic is_odd/1.

is_even(0).
is_even(N) :- N > 0, N1 is N - 1, is_odd(N1).

is_odd(1).
is_odd(N) :- N > 1, N1 is N - 1, is_even(N1).

### 두 서술어에 대한 호출 그래프 구축

In [ ]:
% Build call graph for both predicates
build_call_graph([is_even/1, is_odd/1], _Graph),
format('Call Graph for is_even/1 and is_odd/1:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### 강결합 컴포넌트 (SCC) 찾기

In [ ]:
% Rebuild the graph because variables do not persist between notebook cells
build_call_graph([is_even/1, is_odd/1], _Graph),
% Find SCCs using Tarjan's algorithm
find_sccs(_Graph, _SCCs),
format('Strongly Connected Components:~n'),
forall(member(_SCC, _SCCs),
    format('  ~w~n', [_SCC])).

### SCC가 자명(Trivial)한지 확인

In [ ]:
% Rebuild the derived values so this cell also runs independently
build_call_graph([is_even/1, is_odd/1], _Graph),
find_sccs(_Graph, _SCCs),
% Check each SCC
forall(member(_SCC, _SCCs),
    (   is_trivial_scc(_SCC) ->
        format('  ~w is trivial (single predicate)~n', [_SCC])
    ;
        format('  ~w is NON-TRIVIAL (mutual recursion!)~n', [_SCC])
    )
).

## 예제 3: 복잡한 호출 그래프

여러 서술어가 포함된 더 복잡한 시스템을 분석해 보겠습니다.

In [ ]:
% Define a small program with multiple predicates
:- dynamic grandparent/2.
:- dynamic sibling/2.
:- dynamic cousin/2.

% grandparent uses parent
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

% sibling: same parent, different children
sibling(X, Y) :- parent(P, X), parent(P, Y), X \= Y.

% cousin: parents are siblings
cousin(X, Y) :- parent(P1, X), parent(P2, Y), sibling(P1, P2).

### 전체 호출 그래프 구축

In [ ]:
% Build call graph for all predicates
build_call_graph([grandparent/2, sibling/2, cousin/2], _Graph),
format('Complete Call Graph:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### 서술어 그룹 찾기

시작 서술어를 포함하는 상호 재귀 서술어 그룹을 찾습니다.

In [ ]:
% Find the mutually recursive group containing cousin/2
predicates_in_group(cousin/2, _Group),
format('Mutually recursive group containing cousin/2: ~w~n', [_Group]).

## 예제 4: 패턴 감지

패턴 매처를 사용하여 재귀 유형을 분석해 보겠습니다.

In [ ]:
% Define various recursive patterns
:- dynamic count/3.     % Tail recursive
:- dynamic factorial/2. % Linear recursive
:- dynamic fib/2.       % Tree recursive (or linear if detected)

% Tail recursive count
count([], Acc, Acc).
count([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count(T, Acc1, N).

% Linear recursive factorial
factorial(0, 1).
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),
    F is N * F1.

% Fibonacci (can be detected as linear or tree)
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.

### 꼬리 재귀 감지

In [ ]:
% Check if count/3 is tail recursive
(is_tail_recursive_accumulator(count/3, _AccInfo) ->
    format('✓ count/3 is tail recursive: ~w~n', [_AccInfo])
;
    writeln('✗ count/3 is not tail recursive')
).

### 선형 재귀 감지

In [ ]:
% Check if factorial/2 is linear recursive
(is_linear_recursive_streamable(factorial/2) ->
    writeln('✓ factorial/2 is linear recursive')
;
    writeln('✗ factorial/2 is not linear recursive')
).

### 재귀 호출 횟수 계산

In [ ]:
% Count recursive calls in fibonacci
functor(_FibHead, fib, 2),
user:clause(_FibHead, _FibBody),
once(contains_call_to(_FibBody, fib)),
count_recursive_calls(_FibBody, fib, _Count),
format('Fibonacci body has ~w recursive calls~n', [_Count]).

## DOT 형식을 이용한 시각화

호출 그래프의 Graphviz DOT 표현을 생성해 보겠습니다.

In [ ]:
% Helper to generate DOT format
generate_dot(Graph, DotCode) :-
    findall(Line,
        (   member(From -> To, Graph),
            format(atom(Line), '  "~w" -> "~w";', [From, To])
        ),
        Lines),
    atomic_list_concat(['digraph CallGraph {', '  rankdir=LR;' | Lines], '\n', Body),
    format(atom(DotCode), '~w~n}~n', [Body]).

% Generate DOT for even/odd graph
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
writeln('DOT Code for even/odd call graph:'),
writeln(_DotCode).

### DOT 파일 저장

In [ ]:
% Rebuild the DOT source because variables do not persist between cells
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
setup_call_cleanup(
    open('/shared/data/even_odd_graph.dot', write, _Stream),
    write(_Stream, _DotCode),
    close(_Stream)),
writeln('✓ Saved to /shared/data/even_odd_graph.dot').

writeln('To visualize, run:'),
writeln('  dot -Tpng /shared/data/even_odd_graph.dot -o even_odd_graph.png').

## 연습 과제: 자신만의 코드 분석하기

자신만의 서술어를 정의하고 분석해 보세요!

In [ ]:
% Define your predicates here
% Then build call graphs, find SCCs, and detect patterns

% Example:
% :- dynamic my_predicate/2.
% my_predicate(...) :- ...

% build_call_graph([my_predicate/2], Graph).


## 요약

이 노트북에서 배운 내용:

✅ Prolog 코드로부터 호출 그래프를 구축하는 방법

✅ 상호 재귀를 위한 강결합 컴포넌트(SCC)를 감지하는 방법

✅ 패턴 매처를 사용하여 재귀 유형을 분류하는 방법

✅ 서술어 의존성을 분석하는 방법

✅ DOT 형식으로 호출 그래프를 시각화하는 방법

## 고급 주제

심화 분석을 위한 주제:

- **위상 정렬 (Topological Ordering)**: `topological_order/2`를 사용하여 의존성에 따라 SCC 정렬
- **커스텀 패턴 매처**: 고유한 패턴 감지 서술어 작성
- **누산기 패턴 추출**: 상세 분석을 위해 `extract_accumulator_pattern/2` 사용
- **선형 재귀 금지**: `forbid_linear_recursion/1`을 사용하여 다른 컴파일 전략 강제

## 참고 자료 및 관련 파일

- 제10장: Prolog 인트로스펙션과 이론
- `src/unifyweaver/core/advanced/call_graph.pl`
- `src/unifyweaver/core/advanced/scc_detection.pl`
- `src/unifyweaver/core/advanced/pattern_matchers.pl`